In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

import pandas as pd
from dl2_reports import DL2Report

# `Visual.copy()` and `Visual.get_value()` Examples

This notebook demonstrates two utility methods on `Visual`:

- **`visual.copy()`** — creates a new `Visual` with the same `type`, `dataset_id`, props, and
  `other_elements`, but a fresh unique ID. Useful for stamping the same chart configuration
  into multiple rows/pages without re-specifying every kwarg.

- **`visual.get_value()`** — retrieves a single scalar value from the visual's backing
  DataFrame. Requires the visual to be part of the report tree and its props to include
  `row_index` and `value_column`.

In [16]:
# Sample regional sales data
sales_df = pd.DataFrame({
    "region":  ["North", "South", "East", "West"],
    "revenue": [142_500, 98_300, 175_200, 110_800],
    "units":   [1_420,   980,    1_750,   1_105],
})

report = DL2Report(title="copy() and get_value() demo", compress_visuals=False)
report.add_df("sales", sales_df, format="records", compress=False)

sales_df

c:\Users\kameron\Documents\Projects\Web Projects\superbabycart\datalys2-reporting-python-api\dl2_reports\report.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sample)


,region,revenue,units
0,North,142500,1420
1,South,98300,980
2,East,175200,1750
3,West,110800,1105


## `visual.copy()`

Build a KPI visual once, then use `.copy()` to place the same configuration into
multiple slots. Each copy receives a new unique `id` so the report renderer treats
them as independent components.

Copies can be added back into any layout row via
`row.add_visual(copy.type, visual=copy)`.

In [17]:
page = report.add_page("copy() demo")

# --- Create the original KPI visual (North, row 0) and add it to the first row ---
row1 = page.add_row()
original_kpi = row1.add_kpi(
    dataset_id="sales",
    value_column="revenue",
    row_index=0,
    title="Revenue – North",
    format="currency",
    good_direction="higher",
)

# --- Stamp copies for the remaining regions, overriding only what changes ---
south_kpi = original_kpi.copy()
south_kpi.props["row_index"] = 1
south_kpi.props["title"]     = "Revenue – South"

east_kpi = original_kpi.copy()
east_kpi.props["row_index"] = 2
east_kpi.props["title"]     = "Revenue – East"

west_kpi = original_kpi.copy()
west_kpi.props["row_index"] = 3
west_kpi.props["title"]     = "Revenue – West"

# Add all three copies to a second row using add_visual(type, visual=...)
row2 = page.add_row()
row2.add_visual(south_kpi.type, visual=south_kpi)
row2.add_visual(east_kpi.type,  visual=east_kpi)
row2.add_visual(west_kpi.type,  visual=west_kpi)

# Each copy has its own unique ID
print(f"original : {original_kpi.id}")
print(f"south    : {south_kpi.id}")
print(f"east     : {east_kpi.id}")
print(f"west     : {west_kpi.id}")
all_unique = len({original_kpi.id, south_kpi.id, east_kpi.id, west_kpi.id}) == 4
print(f"All IDs unique: {all_unique}")

original : elem-37
south    : elem-38
east     : elem-39
west     : elem-40
All IDs unique: True


## `visual.get_value()`

`get_value()` reads back the scalar that a visual represents directly from the
underlying DataFrame. This is useful for:
- Programmatic thresholds / conditional logic after building the report layout
- Logging or asserting values without re-querying the DataFrame manually

Requirements:
1. The visual must be part of the report tree (i.e. added to a row).
2. Its props must include `row_index` and `value_column`.

In [18]:
# get_value() works on the original and all copies (all are in the report tree)
print("Revenue by region")
print("-" * 30)
for kpi, region in [
    (original_kpi, "North"),
    (south_kpi,    "South"),
    (east_kpi,     "East"),
    (west_kpi,     "West"),
]:
    value = kpi.get_value()
    print(f"  {region:<6}: ${value:>10,.0f}")

# Useful for conditional logic – e.g. flag underperforming regions
TARGET = 120_000
print(f"\nTarget: ${TARGET:,}")
for kpi, region in [
    (original_kpi, "North"),
    (south_kpi,    "South"),
    (east_kpi,     "East"),
    (west_kpi,     "West"),
]:
    value = kpi.get_value()
    flag = "above target" if value >= TARGET else "BELOW TARGET"
    print(f"  {region:<6}: {flag}")

Revenue by region
------------------------------
  North : $   142,500
  South : $    98,300
  East  : $   175,200
  West  : $   110,800

Target: $120,000
  North : above target
  South : BELOW TARGET
  East  : above target
  West  : BELOW TARGET


## Combined: loop-driven layout with `copy()` + `get_value()`

A common pattern is to create a prototype visual, then loop over rows to stamp
copies into a layout while simultaneously reading back each value for summary
calculations.

In [19]:
from typing import cast

from dl2_reports.components.visual import Visual

summary_page = report.add_page("Summary")

# --- Build a prototype by adding the first KPI normally ---
kpi_row = summary_page.add_row()
regions = sales_df["region"].tolist()

proto = cast(Visual, kpi_row.add_kpi(
    dataset_id="sales",
    value_column="units",
    row_index=0,
    title=f"Units – {regions[0]}",
    format="number",
))

# Stamp copies for each remaining region and collect all values
kpi_visuals = [proto]
for i, region in enumerate(regions[1:], start=1):
    kpi_copy = proto.copy()
    kpi_copy.props["row_index"] = i
    kpi_copy.props["title"]     = f"Units – {region}"
    kpi_row.add_visual(kpi_copy.type, visual=kpi_copy)
    kpi_visuals.append(kpi_copy)

# get_value() to compute a grand total without touching the DataFrame directly
total_units = sum(kpi.get_value() for kpi in kpi_visuals)

print(f"Total units (via get_value) : {total_units:,}")
print(f"Total units (DataFrame sum) : {sales_df['units'].sum():,}")
print(f"Match: {total_units == sales_df['units'].sum()}")

Total units (via get_value) : 5,255
Total units (DataFrame sum) : 5,255
Match: True


## `report.get_value()` + `on_condition()` — conditional element inclusion

`report.get_value(dataset_name, column, row_index)` queries a value directly
from a registered dataset (by name) without needing a visual reference.
Combined with `row.on_condition(bool)`, you can conditionally add visuals to a
row in a single chained call — no `if/else` blocks needed.

Signature: `report.get_value(data_source_name, column_name, row_index=-1)`

The example below reads each region's revenue and uses `on_condition()` to:
- Add a highlight card **only when** the best region exceeds a high-water mark
- Add a warning card per region **only when** that region's revenue is below target

In [20]:
TARGET = 120_000

conditional_page = report.add_page("Conditional Layout")

# --- Bar chart of all regions ---
bar_row = conditional_page.add_row()
bar_row.add_bar(
    dataset_id="sales",
    x_column="region",
    y_columns=["revenue"],
    title="Revenue by Region",
)

# Use report.get_value() to check the top performer and add a highlight card
# only when the best region exceeds a high-water mark.
HIGH_WATER = 150_000
revenues = [report.get_value("sales", "revenue", i) for i in range(len(sales_df))]
best_idx = revenues.index(max(revenues))
best_region = report.get_value("sales", "region", best_idx)
best_revenue = revenues[best_idx]

print(f"Top performer: {best_region} at ${best_revenue:,} (high-water: ${HIGH_WATER:,})")

highlight_row = conditional_page.add_row()
highlight_row.on_condition(best_revenue >= HIGH_WATER).add_card(
    title="Top Performer",
    text=f"**{best_region}** leads with **${best_revenue:,}** in revenue — above the ${HIGH_WATER:,} high-water mark.",
    content_type="md",
)
print(f"  -> {'Highlight card added' if best_revenue >= HIGH_WATER else 'No highlight card (no region exceeded high-water mark)'}")

# --- Warning cards — one per underperforming region ---
warning_row = conditional_page.add_row()

for i, region_name in enumerate(sales_df["region"]):
    region_revenue = report.get_value("sales", "revenue", i)
    warning_row.on_condition(region_revenue < TARGET).add_card(
        title=f"Warning: {region_name} below target",
        text=f"Revenue ${region_revenue:,.0f} is ${TARGET - region_revenue:,.0f} short of the ${TARGET:,} target.",
        content_type="md",
    )
    if region_revenue < TARGET:
        print(f"  -> Warning card added for {region_name} (${region_revenue:,})")
    else:
        print(f"  -> {region_name} OK (${region_revenue:,})")

Top performer: East at $175,200 (high-water: $150,000)
  -> Highlight card added
  -> North OK ($142,500)
  -> Warning card added for South ($98,300)
  -> East OK ($175,200)
  -> Warning card added for West ($110,800)


In [21]:
report.save("copy_and_get_value.html")
report.show()

## `get_value()` with `compress=True`

Verify that `report.get_value()` works correctly when a dataset was added with `compress=True`.
Previously, the compressed path cleared `data` to `[]`, causing a "column not found" error.
This test asserts the fix is in place.

In [22]:
# Build a fresh report with compress=True on the dataset
compressed_report = DL2Report(title="get_value compress test", compress_visuals=False)
compressed_report.add_df("sales_compressed", sales_df, format="records", compress=True)

# get_value should work for every column/row even though data[] was cleared
tests = [
    ("region",  0, "North"),
    ("revenue", 1, 98_300),
    ("units",   2, 1_750),
    ("region",  -1, "West"),   # negative index (last row)
]

all_passed = True
for col, idx, expected in tests:
    result = compressed_report.get_value("sales_compressed", col, idx)
    status = "PASS" if result == expected else f"FAIL (got {result!r}, expected {expected!r})"
    if status != "PASS":
        all_passed = False
    print(f"get_value('sales_compressed', {col!r}, {idx}) => {result!r}  [{status}]")

print()
print("All tests passed!" if all_passed else "SOME TESTS FAILED")

get_value('sales_compressed', 'region', 0) => 'North'  [PASS]
get_value('sales_compressed', 'revenue', 1) => np.int64(98300)  [PASS]
get_value('sales_compressed', 'units', 2) => np.int64(1750)  [PASS]
get_value('sales_compressed', 'region', -1) => 'West'  [PASS]

All tests passed!


c:\Users\kameron\Documents\Projects\Web Projects\superbabycart\datalys2-reporting-python-api\dl2_reports\report.py:154: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sample)
